# TradeSense · Component 03 · TGNN dataset audit (READ-ONLY) → `audit_tgnn_dataset_v2`
Dataset: `MaximumLeverage/crypto-lob-stream` (Binance, BTCUSDT and ETHUSDT in scope, SOLUSDT ignored), June–July 2026, Parquet.

**This notebook does not train anything and does not write inside the dataset folder.** It reads the files, measures what is really there and prints a FINAL AUDIT SHARE BLOCK that answers: files/schemas/rows; timestamp coverage, ordering, gaps; whether snapshots + depth updates reconstruct a valid book; whether order IDs / add-cancel-execute events exist or only price-level updates; which signals can support spoofing-like, layering-like and liquidity-withdrawal analysis; whether verified manipulation labels exist; provenance and licence; a feasible causal graph design; compatibility with the 2024 Layer 1 data; and whether the dataset is sufficient for a defensible TGNN experiment.

Design rules: the schema is **not assumed** (roles are inferred from column names and types, and printed for review); every section is wrapped so a failure is recorded and the rest still runs; the dataset folder is SHA-256 pinned before and after to prove it was not modified; SOLUSDT files are listed by name/size only and never opened; outputs go to a new staging folder renamed to `audit_tgnn_dataset_v2` at the end. Numbers marked *heuristic* are descriptive proxies, not labels or evidence of manipulation.

## 0 · Setup, inventory and read-only pinning

In [ ]:
import os, sys, json, re, time, math, hashlib, uuid, warnings, bisect, heapq
from pathlib import Path
from datetime import datetime, timezone
from collections import defaultdict, Counter
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from IPython.display import display
warnings.filterwarnings('ignore')
pd.set_option('display.width', 230); pd.set_option('display.max_columns', 60); pd.set_option('display.max_rows', 150); pd.set_option('display.max_colwidth', 80)
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not in Colab -> using TRADESENSE_BASE.')
BASE = Path(os.environ.get('TRADESENSE_BASE', '/content/drive/MyDrive/research datasets'))
TG = BASE / 'TGNN_Public_OrderBook_Dataset'; L1 = BASE / 'processed_layer1_final_v1'; OUT_FINAL = BASE / 'audit_tgnn_dataset_v2'; STAGE = BASE / ('.tgnn_audit_stage_' + uuid.uuid4().hex[:8])
assert TG.is_dir(), f'dataset folder missing: {TG}'; assert not OUT_FINAL.exists(), f'REFUSING to overwrite {OUT_FINAL}'
UTC = lambda: datetime.now(timezone.utc).isoformat(); RUN_STARTED = UTC(); OPENED = []; SECTION_ERRORS = {}; REPORT = {}
def sha256_file(p, chunk=16 << 20):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(chunk), b''): h.update(b)
    return h.hexdigest()
def jsonable(o):
    if isinstance(o, np.integer): return int(o)
    if isinstance(o, np.floating): return None if not np.isfinite(o) else float(o)
    if isinstance(o, np.bool_): return bool(o)
    if isinstance(o, np.ndarray): return o.tolist()
    if isinstance(o, (pd.Timestamp, datetime)): return o.isoformat()
    return str(o)
def dump_json(obj, path):
    path.parent.mkdir(parents=True, exist_ok=True); path.write_text(json.dumps(obj, indent=2, default=jsonable), encoding='utf-8')
def section(name):
    def deco(fn):
        try: fn()
        except Exception as e:
            import traceback; SECTION_ERRORS[name] = traceback.format_exc()[-1500:]; print(f'!! SECTION FAILED: {name}: {type(e).__name__}: {e}')
        return fn
    return deco
def opened(p): OPENED.append(str(p))
t0 = time.time()
TG_FILES = sorted(p for p in TG.rglob('*') if p.is_file())
PIN_TG = {}
for p in TG_FILES: PIN_TG[str(p.relative_to(TG))] = {'size': p.stat().st_size, 'mtime_ns': p.stat().st_mtime_ns, 'sha256': sha256_file(p)}
PIN_L1 = {str(p.relative_to(L1)): {'size': p.stat().st_size, 'mtime_ns': p.stat().st_mtime_ns} for p in sorted(L1.rglob('*')) if p.is_file()} if L1.is_dir() else {}
print(f'pinned {len(PIN_TG)} dataset files (sha256) in {time.time()-t0:.0f}s; Layer 1 files pinned by size/mtime: {len(PIN_L1)}')

SYM_RE = re.compile(r'(btc|eth|sol|bnb|xrp|doge|ada)[-_/]?usdt', re.I)
def sym_of(rel):
    m = SYM_RE.search(rel); return (m.group(1).upper() + 'USDT') if m else None
def cat_of(rel):
    l = rel.lower()
    if 'trade' in l: return 'trades'
    if 'snap' in l: return 'snapshot'
    if any(k in l for k in ('depth', 'diff', 'update', 'delta')): return 'depth_update'
    if any(k in l for k in ('book', 'lob', 'l2', 'orderbook')): return 'book_unspecified'
    return 'other'
INV = []
for p in TG_FILES:
    rel = str(p.relative_to(TG)); sy = sym_of(rel); INV.append({'path': rel, 'size_bytes': p.stat().st_size, 'size_mb': round(p.stat().st_size / 1e6, 2), 'ext': p.suffix.lower(), 'symbol_from_path': sy, 'category_from_path': cat_of(rel) if p.suffix.lower() == '.parquet' else 'non-parquet', 'hidden': any(x.startswith('.') for x in p.relative_to(TG).parts), 'in_scope': (not any(x.startswith('.') for x in p.relative_to(TG).parts)) and ((sy in ('BTCUSDT', 'ETHUSDT')) or (sy is None and p.suffix.lower() == '.parquet'))})
INVDF = pd.DataFrame(INV); display(INVDF[~INVDF.hidden])
print('hidden/hf-cache metadata files (pinned, never analysed):', int(INVDF.hidden.sum()))
n_pq = int((INVDF.ext == '.parquet').sum()); INVDF.loc[INVDF.hidden, 'category_from_path'] = 'hf_cache_metadata'
for r_ in INV:
    if r_['hidden']: r_['category_from_path'] = 'hf_cache_metadata'
tot = INVDF.size_bytes.sum()
print(f'files: {len(INVDF)} total (parquet {n_pq}) | total size {tot/1e9:.3f} GB | user states 18 files / ~1.605 GB')
print('out-of-scope data files (never opened):', INVDF[(~INVDF.in_scope) & (~INVDF.hidden) & (INVDF.ext == '.parquet')].path.tolist())
REPORT['inventory'] = INV; REPORT['expected'] = {'files': 18, 'gb': 1.605}; REPORT['observed'] = {'files': len(INVDF), 'parquet': n_pq, 'gb': tot / 1e9}

Mounted at /content/drive
pinned 63 dataset files (sha256) in 65s; Layer 1 files pinned by size/mtime: 400


,path,size_bytes,size_mb,ext,symbol_from_path,category_from_path,hidden,in_scope
44,README.md,6203,0.01,.md,None,non-parquet,False,False
45,depth/binance/BTCUSDT/2026-06.parquet,385932069,385.93,.parquet,BTCUSDT,depth_update,False,True
46,depth/binance/BTCUSDT/2026-07.parquet,317605448,317.61,.parquet,BTCUSDT,depth_update,False,True
47,depth/binance/ETHUSDT/2026-06.parquet,312208107,312.21,.parquet,ETHUSDT,depth_update,False,True
48,depth/binance/ETHUSDT/2026-07.parquet,282650449,282.65,.parquet,ETHUSDT,depth_update,False,True
49,depth/binance/SOLUSDT/2026-06.parquet,110365893,110.37,.parquet,SOLUSDT,depth_update,False,False
50,depth/binance/SOLUSDT/2026-07.parquet,82871592,82.87,.parquet,SOLUSDT,depth_update,False,False
51,snapshots/binance/BTCUSDT/2026-06.parquet,299002,0.30,.parquet,BTCUSDT,snapshot,False,True
52,snapshots/binance/BTCUSDT/2026-07.parquet,350331,0.35,.parquet,BTCUSDT,snapshot,False,True
53,snapshots/binance/ETHUSDT/2026-06.parquet,240589,0.24,.parquet,ETHUSDT,snapshot,False,True


hidden/hf-cache metadata files (pinned, never analysed): 44
files: 63 total (parquet 18) | total size 1.724 GB | user states 18 files / ~1.605 GB
out-of-scope data files (never opened): ['depth/binance/SOLUSDT/2026-06.parquet', 'depth/binance/SOLUSDT/2026-07.parquet', 'snapshots/binance/SOLUSDT/2026-06.parquet', 'snapshots/binance/SOLUSDT/2026-07.parquet', 'trades/binance/SOLUSDT/2026-06.parquet', 'trades/binance/SOLUSDT/2026-07.parquet']


## 1 · Parquet metadata (footers only) and column-role inference

In [ ]:
SCOPE = [r for r in INV if r['ext'] == '.parquet' and r['in_scope']]; META = {}; SCHEMA = {}
for r in SCOPE:
    p = TG / r['path']; opened(p); pf = pq.ParquetFile(p); md_ = pf.metadata; sch = pf.schema_arrow
    META[r['path']] = {'rows': md_.num_rows, 'row_groups': md_.num_row_groups, 'columns': md_.num_columns, 'created_by': md_.created_by, 'compression': md_.row_group(0).column(0).compression if md_.num_row_groups else None,
                       'kv_metadata_keys': sorted((k.decode() if isinstance(k, bytes) else k) for k in (sch.metadata or {}).keys()), 'schema': {n: str(sch.field(n).type) for n in sch.names}}
    SCHEMA[r['path']] = sch
MT = pd.DataFrame([{'path': k, 'rows': v['rows'], 'row_groups': v['row_groups'], 'cols': v['columns'], 'compression': v['compression'], 'created_by': v['created_by']} for k, v in META.items()]); display(MT)
for k, v in META.items(): print(k, '->', v['schema'])
def infer_roles(sch):
    R = defaultdict(list); exact = {'E': 'time', 'T': 'time', 'U': 'first_id', 'u': 'final_id', 'pu': 'prev_id', 'b': 'nested_bids', 'a': 'nested_asks', 's': 'symbol', 'p': 'price', 'q': 'qty', 'm': 'side'}
    for n in sch.names:
        l = n.lower(); t = sch.field(n).type
        if n in exact: R[exact[n]].append(n); continue
        if pa.types.is_list(t) or pa.types.is_large_list(t) or pa.types.is_struct(t):
            R['nested_bids' if re.search(r'bid', l) else 'nested_asks' if re.search(r'ask', l) else 'nested_other'].append(n); continue
        if re.search(r'(bid|ask).*(px|price|prc|qty|size|amount|vol|quantity)?.*\d+$|^(b|a)(px|qty|sz)_?\d+$|(px|price|qty|size)_?(bid|ask)_?\d+', l): R['wide'].append(n); continue
        if pa.types.is_timestamp(t) or re.search(r'(^|_)(ts|time|timestamp|datetime|date|epoch)(_|$)|event_?time|trade_?time|transact', l): R['time'].append(n)
        if re.search(r'(^|_)(price|px|prc)(_|$)', l): R['price'].append(n)
        if re.search(r'(qty|quantity|size|amount|(^|_)vol(ume)?(_|$))', l): R['qty'].append(n)
        if re.search(r'(^|_)side(_|$)|buyer_?maker|is_?bid|taker|aggressor', l): R['side'].append(n)
        if re.search(r'first.*(id|update)|first_?u|^u_?first', l): R['first_id'].append(n)
        if re.search(r'(final|last).*(id|update)|lastupdateid|^u_?last', l): R['final_id'].append(n)
        if re.search(r'^prev|^pu$|prev.*(id|update)', l): R['prev_id'].append(n)
        if re.search(r'trade_?id|agg.*id|^id$', l): R['trade_id'].append(n)
        if re.search(r'order_?id|oid|client_?order', l): R['order_id'].append(n)
        if re.search(r'(event|action|op|update)_?type|^type$|^action$|^event$|msg_?type', l): R['event_type'].append(n)
        if re.search(r'symbol|^pair$|^s$', l): R['symbol'].append(n)
        if re.search(r'label|manip|spoof|layer|wash|anomal|suspic|fraud|flag|abuse|fake', l): R['label_like'].append(n)
    return dict(R)
ROLES = {k: infer_roles(s) for k, s in SCHEMA.items()}
for k, v in ROLES.items(): print(k, '->', v)
def layout_of(sch, roles, cat):
    if cat == 'trades' or ('price' in roles and 'qty' in roles and 'side' in roles and 'first_id' not in roles and 'final_id' not in roles and cat != 'depth_update' and 'nested_bids' not in roles): return 'trades_like'
    if 'nested_bids' in roles or 'nested_asks' in roles: return 'nested'
    if 'wide' in roles and len(roles['wide']) >= 4: return 'wide'
    if 'price' in roles and 'qty' in roles: return 'long'
    return 'unknown'
LAYOUT = {k: layout_of(SCHEMA[k], ROLES[k], next(r['category_from_path'] for r in SCOPE if r['path'] == k)) for k in META}
for r in SCOPE: r['rows'] = META[r['path']]['rows']; r['layout'] = LAYOUT[r['path']]
print('layouts:', Counter(LAYOUT.values())); REPORT['metadata'] = META; REPORT['roles'] = ROLES; REPORT['layout'] = LAYOUT

,path,rows,row_groups,cols,compression,created_by
0,depth/binance/BTCUSDT/2026-06.parquet,68905191,66,8,SNAPPY,parquet-cpp-arrow version 24.0.0
1,depth/binance/BTCUSDT/2026-07.parquet,50583603,49,8,SNAPPY,parquet-cpp-arrow version 24.0.0
2,depth/binance/ETHUSDT/2026-06.parquet,45852893,44,8,SNAPPY,parquet-cpp-arrow version 24.0.0
3,depth/binance/ETHUSDT/2026-07.parquet,38398343,37,8,SNAPPY,parquet-cpp-arrow version 24.0.0
4,snapshots/binance/BTCUSDT/2026-06.parquet,34000,1,7,SNAPPY,parquet-cpp-arrow version 24.0.0
5,snapshots/binance/BTCUSDT/2026-07.parquet,42000,1,7,SNAPPY,parquet-cpp-arrow version 24.0.0
6,snapshots/binance/ETHUSDT/2026-06.parquet,32000,1,7,SNAPPY,parquet-cpp-arrow version 24.0.0
7,snapshots/binance/ETHUSDT/2026-07.parquet,42000,1,7,SNAPPY,parquet-cpp-arrow version 24.0.0
8,trades/binance/BTCUSDT/2026-06.parquet,8118466,8,7,SNAPPY,parquet-cpp-arrow version 24.0.0
9,trades/binance/BTCUSDT/2026-07.parquet,6363416,7,7,SNAPPY,parquet-cpp-arrow version 24.0.0


depth/binance/BTCUSDT/2026-06.parquet -> {'timestamp_ms': 'int64', 'asset': 'string', 'side': 'string', 'price': 'double', 'quantity': 'double', 'first_update_id': 'int64', 'last_update_id': 'int64', 'exchange': 'string'}
depth/binance/BTCUSDT/2026-07.parquet -> {'timestamp_ms': 'int64', 'asset': 'string', 'side': 'string', 'price': 'double', 'quantity': 'double', 'first_update_id': 'int64', 'last_update_id': 'int64', 'exchange': 'string'}
depth/binance/ETHUSDT/2026-06.parquet -> {'timestamp_ms': 'int64', 'asset': 'string', 'side': 'string', 'price': 'double', 'quantity': 'double', 'first_update_id': 'int64', 'last_update_id': 'int64', 'exchange': 'string'}
depth/binance/ETHUSDT/2026-07.parquet -> {'timestamp_ms': 'int64', 'asset': 'string', 'side': 'string', 'price': 'double', 'quantity': 'double', 'first_update_id': 'int64', 'last_update_id': 'int64', 'exchange': 'string'}
snapshots/binance/BTCUSDT/2026-06.parquet -> {'timestamp_ms': 'int64', 'asset': 'string', 'side': 'string', 'pri

## 2 · Provenance and licence text shipped with the download (README / LICENSE / metadata files)

In [ ]:
PROV = {'text_files': {}, 'license_lines': [], 'urls': [], 'yaml_front_matter': None}
@section('provenance')
def _prov():
    for r in INV:
        if r['ext'] in ('.md', '.txt', '.json', '.yaml', '.yml', '.csv', '') or Path(r['path']).name.lower().startswith(('readme', 'license', 'licence', 'citation')):
            if r['ext'] == '.parquet' or r['size_bytes'] > 2_000_000 or r['hidden']: continue
            p = TG / r['path']; opened(p); txt = p.read_text(encoding='utf-8', errors='replace'); PROV['text_files'][r['path']] = txt[:14000]
            for ln in txt.splitlines():
                if re.search(r'licen[sc]e|terms of use|copyright|cc-by|mit\b|apache|odbl', ln, re.I): PROV['license_lines'].append((r['path'], ln.strip()[:200]))
            PROV['urls'] += re.findall(r'https?://[^\s)>\]]+', txt)[:20]
            m = re.match(r'---\n(.*?)\n---', txt, re.S)
            if m and PROV['yaml_front_matter'] is None: PROV['yaml_front_matter'] = m.group(1)[:3000]
    kv = {k: v['kv_metadata_keys'] for k, v in META.items() if v['kv_metadata_keys'] and v['kv_metadata_keys'] != ['pandas']}
    PROV['parquet_custom_metadata_keys'] = kv; PROV['created_by'] = sorted({v['created_by'] for v in META.values() if v['created_by']})
    print('text/metadata files found:', list(PROV['text_files'])); print('licence lines:', PROV['license_lines'][:6]); print('urls:', PROV['urls'][:6]); print('parquet created_by:', PROV['created_by'])
    if PROV['yaml_front_matter']: print('--- YAML front matter ---\n' + PROV['yaml_front_matter'][:1500])
REPORT['provenance'] = PROV

text/metadata files found: ['README.md']
licence lines: [('README.md', 'license: mit'), ('README.md', 'Full depth-of-book data is paywalled or licensed for most asset classes; crypto'), ('README.md', '([GitHub](https://github.com/Goodie-Goody/crypto_lob_stream_pypi), MIT licensed,')]
urls: ['https://pypi.org/project/crypto-lob-stream/', 'https://dev.to/oliverzehentleitner/your-binance-l2-order-book-can-be-gap-free-and-still-be-wrong-1kk8', 'https://pypi.org/project/crypto-lob-stream/', 'https://github.com/Goodie-Goody/crypto_lob_stream_pypi', 'https://pypi.org/project/crypto-lob-stream/', 'https://github.com/Goodie-Goody/crypto_lob_stream_pypi']
parquet created_by: ['parquet-cpp-arrow version 24.0.0']
--- YAML front matter ---
license: mit
task_categories:
  - time-series-forecasting
tags:
  - finance
  - cryptocurrency
  - order-book
  - market-microstructure
  - high-frequency
pretty_name: Binance BTC/ETH/SOL L2 Order Book & Trades
configs:
  - config_name: depth
    data_files: "dep

## 3 · Timestamp coverage, ordering, gaps, sampling frequency, update-ID continuity (streamed; no file is loaded whole)

In [ ]:
import pyarrow.compute as pc
BINS = np.logspace(3, 14.5, 70); GAP_T = {'>1s': 1e9, '>10s': 1e10, '>60s': 6e10, '>300s': 3e11}
def to_ns(arr, st):
    t = arr.type
    if pa.types.is_timestamp(t):
        v = pc.fill_null(arr.cast(pa.timestamp('ns', tz=t.tz)).cast(pa.int64()), -1); return v.to_numpy(zero_copy_only=False).astype(np.int64)
    if arr.null_count == 0 and pa.types.is_integer(t): v = arr.to_numpy(zero_copy_only=False).astype(np.int64)
    else: v = np.asarray(pc.fill_null(arr.cast(pa.float64()), float('nan')).to_numpy(zero_copy_only=False), dtype='float64')
    if st.get('mult') is None:
        med = float(np.nanmedian(v.astype('float64'))); st['mult'] = 10 ** 9 if med < 1e11 else (10 ** 6 if med < 1e14 else (10 ** 3 if med < 1e17 else 1)); st['unit'] = {10 ** 9: 's', 10 ** 6: 'ms', 10 ** 3: 'us', 1: 'ns'}[st['mult']]
    if v.dtype.kind == 'f': ok = np.isfinite(v); out = np.full(len(v), -1, np.int64); out[ok] = (v[ok] * st['mult']).astype(np.int64); return out
    return v * st['mult']
class TimeAcc:
    def __init__(s): s.n = 0; s.valid = 0; s.mn = None; s.mx = None; s.neg = 0; s.zero = 0; s.last = None; s.hist = np.zeros(len(BINS) - 1, np.int64); s.top = []; s.days = Counter(); s.hours = set(); s.gapc = Counter(); s.st = {}
    def add(s, ns):
        s.n += len(ns); ns = ns[ns > 0]; s.valid += len(ns)
        if len(ns) == 0: return
        s.mn = int(ns.min()) if s.mn is None else min(s.mn, int(ns.min())); s.mx = int(ns.max()) if s.mx is None else max(s.mx, int(ns.max()))
        x = ns if s.last is None else np.r_[s.last, ns]; d = np.diff(x); s.last = int(ns[-1]); s.neg += int((d < 0).sum()); s.zero += int((d == 0).sum()); pos = d[d > 0]
        if len(pos):
            s.hist += np.histogram(pos, bins=BINS)[0]
            for k, th in GAP_T.items(): s.gapc[k] += int((pos > th).sum())
            big = np.flatnonzero(d > 6e10)
            for i in big[:50]: heapq.heappush(s.top, (int(d[i]), int(x[i]))); s.top = heapq.nlargest(20, s.top)
        dd = pd.Series(ns // 86_400_000_000_000).value_counts()
        for k, v in dd.items(): s.days[int(k)] += int(v)
        for h in np.unique(ns // 3_600_000_000_000): s.hours.add(int(h))
    def pct(s, q):
        c = np.cumsum(s.hist);
        if c[-1] == 0: return None
        return float(BINS[1:][np.searchsorted(c, q * c[-1])])
    def summary(s):
        iso = lambda v: None if v is None else pd.Timestamp(v, unit='ns', tz='UTC').isoformat()
        span_h = None if s.mn is None else (s.mx - s.mn) / 3.6e12
        return {'rows': s.n, 'valid_time_rows': s.valid, 'unit_detected': s.st.get('unit'), 'tmin': iso(s.mn), 'tmax': iso(s.mx), 'span_hours': span_h, 'inversions_(next<prev)': s.neg, 'inversion_share': s.neg / max(s.valid - 1, 1), 'equal_consecutive_share': s.zero / max(s.valid - 1, 1),
                'gap_ms_median_approx': (s.pct(0.5) or 0) / 1e6, 'gap_ms_p95_approx': (s.pct(0.95) or 0) / 1e6, 'gap_ms_p99_approx': (s.pct(0.99) or 0) / 1e6, 'gaps': dict(s.gapc), 'largest_gaps_s': [(round(g / 1e9, 1), iso(t)) for g, t in s.top[:8]],
                'utc_days_with_data': len(s.days), 'rows_per_day_min_max': (min(s.days.values()), max(s.days.values())) if s.days else None, 'hours_with_data': len(s.hours), 'hours_in_span': None if span_h is None else math.ceil(span_h) + 1, 'days_list': sorted(pd.Timestamp(k * 86_400_000_000_000, unit='ns').strftime('%Y-%m-%d') for k in s.days)}
class IdAcc:
    def __init__(s): s.n_ev = 0; s.contig = 0; s.gap = 0; s.overlap = 0; s.back = 0; s.last_u = None; s.last_key = None; s.pu_ok = 0; s.pu_bad = 0; s.big_gaps = []
    def add(s, U, u, pu=None):
        U = np.asarray(U, np.int64); u = np.asarray(u, np.int64); key = np.stack([U, u], 1); chg = np.ones(len(U), bool); chg[1:] = (key[1:] != key[:-1]).any(axis=1)
        if s.last_key is not None: chg[0] = bool((key[0] != s.last_key).any())
        idx = np.flatnonzero(chg); Ue, ue = U[idx], u[idx]
        prev_u = np.r_[np.array([s.last_u], np.int64) if s.last_u is not None else np.array([-1], np.int64), ue[:-1]] if len(ue) else ue
        ok = prev_u >= 0; s.n_ev += int(ok.sum() if s.last_u is None else len(Ue)); d = Ue - prev_u
        s.contig += int(((d == 1) & ok).sum()); s.gap += int(((d > 1) & ok).sum()); s.overlap += int(((d <= 0) & (ue >= prev_u) & ok).sum()); s.back += int(((ue < prev_u) & ok).sum())
        if pu is not None and len(idx): pu_e = np.asarray(pu, np.int64)[idx]; s.pu_ok += int(((pu_e == prev_u) & ok).sum()); s.pu_bad += int(((pu_e != prev_u) & ok).sum())
        if len(ue): s.last_u = int(ue[-1]); s.last_key = key[-1]
    def summary(s): return {'events': s.n_ev, 'contiguous(U==prev_u+1)': s.contig, 'gaps(U>prev_u+1)': s.gap, 'overlap/repeat': s.overlap, 'backwards(u<prev_u)': s.back, 'pu_equals_prev_u': s.pu_ok, 'pu_not_equal': s.pu_bad}
def primary_time(roles):
    c = roles.get('time', []);
    if not c: return None
    pri = [n for n in c if re.search(r'exch|event|transact|trade|^e$|^t$', n, re.I)]; return (pri or c)[0]
TIMESUM, IDSUM, COLSTATS = {}, {}, {}
@section('time-scan')
def _scan():
    for r in SCOPE:
        path = TG / r['path']; roles = ROLES[r['path']]; pt = primary_time(roles); pf = pq.ParquetFile(path); opened(path)
        cols = [c for c in dict.fromkeys(([pt] if pt else []) + roles.get('first_id', [])[:1] + roles.get('final_id', [])[:1] + roles.get('prev_id', [])[:1])]
        if not cols: continue
        ta = TimeAcc(); ia = IdAcc() if (roles.get('first_id') and roles.get('final_id')) else None; secondary = {n: TimeAcc() for n in roles.get('time', []) if n != pt}
        for b in pf.iter_batches(batch_size=1_000_000, columns=list(dict.fromkeys(cols + list(secondary)))):
            if pt: ta.add(to_ns(b.column(pt), ta.st))
            for n, acc in secondary.items(): acc.add(to_ns(b.column(n), acc.st))
            if ia: ia.add(b.column(roles['first_id'][0]).to_numpy(zero_copy_only=False), b.column(roles['final_id'][0]).to_numpy(zero_copy_only=False), b.column(roles['prev_id'][0]).to_numpy(zero_copy_only=False) if roles.get('prev_id') else None)
        TIMESUM[r['path']] = {'primary_time_column': pt, **(ta.summary() if pt else {})}
        for n, acc in secondary.items(): TIMESUM[r['path']].setdefault('secondary_time', {})[n] = {k: v for k, v in acc.summary().items() if k in ('tmin', 'tmax', 'inversions_(next<prev)', 'gap_ms_median_approx')}
        if ia: IDSUM[r['path']] = ia.summary()
        md_ = pf.metadata; st = {}
        for ci in range(md_.num_columns):
            nm = md_.schema.column(ci).name; nulls = 0; mn = mx = None; have = True
            for rg in range(md_.num_row_groups):
                s = md_.row_group(rg).column(ci).statistics
                if s is None or not s.has_min_max: have = False; break
                nulls += s.null_count or 0; mn = s.min if mn is None else min(mn, s.min); mx = s.max if mx is None else max(mx, s.max)
            if have and mn is not None and not isinstance(mn, (list, dict)): st[nm] = {'min': str(mn)[:40], 'max': str(mx)[:40], 'nulls': nulls}
        COLSTATS[r['path']] = st
    TS_ = pd.DataFrame({k: {x: v.get(x) for x in ('primary_time_column', 'rows', 'tmin', 'tmax', 'span_hours', 'inversions_(next<prev)', 'equal_consecutive_share', 'gap_ms_median_approx', 'gap_ms_p99_approx', 'utc_days_with_data', 'hours_with_data', 'hours_in_span')} for k, v in TIMESUM.items()}).T; display(TS_)
    for k, v in TIMESUM.items(): print(k, '| gaps', v.get('gaps'), '| largest gaps (s, start)', v.get('largest_gaps_s', [])[:4])
    for k, v in IDSUM.items(): print('update-id continuity', k, v)
REPORT['time'] = TIMESUM; REPORT['ids'] = IDSUM; REPORT['column_stats'] = COLSTATS

,primary_time_column,rows,tmin,tmax,span_hours,inversions_(next<prev),equal_consecutive_share,gap_ms_median_approx,gap_ms_p99_approx,utc_days_with_data,hours_with_data,hours_in_span
depth/binance/BTCUSDT/2026-06.parquet,timestamp_ms,68905191,2026-06-04T00:54:40.378000+00:00,2026-06-30T23:58:26.423000+00:00,647.06279,0,0.971797,146.779927,146.779927,27,648,649
depth/binance/BTCUSDT/2026-07.parquet,timestamp_ms,50583603,2026-07-01T00:54:17.803000+00:00,2026-07-31T23:56:46.326000+00:00,743.041256,0,0.955845,146.779927,146.779927,31,744,745
depth/binance/ETHUSDT/2026-06.parquet,timestamp_ms,45852893,2026-06-04T00:54:40.378000+00:00,2026-06-30T23:58:26.424000+00:00,647.062791,0,0.957639,146.779927,146.779927,27,648,649
depth/binance/ETHUSDT/2026-07.parquet,timestamp_ms,38398343,2026-07-01T00:54:17.803000+00:00,2026-07-31T23:56:46.327000+00:00,743.041257,0,0.942001,146.779927,146.779927,31,744,745
snapshots/binance/BTCUSDT/2026-06.parquet,timestamp_ms,34000,2026-06-05T06:29:48.543000+00:00,2026-06-29T08:15:08.598000+00:00,577.755571,0,0.999529,215443469.003188,215443469.003188,14,15,579
snapshots/binance/BTCUSDT/2026-07.parquet,timestamp_ms,42000,2026-07-01T08:35:18.745000+00:00,2026-07-30T17:14:18.706000+00:00,704.649989,0,0.999524,215443469.003188,215443469.003188,16,20,706
snapshots/binance/ETHUSDT/2026-06.parquet,timestamp_ms,32000,2026-06-05T06:29:49.202000+00:00,2026-06-29T08:15:09.268000+00:00,577.755574,0,0.999531,215443469.003188,215443469.003188,14,15,579
snapshots/binance/ETHUSDT/2026-07.parquet,timestamp_ms,42000,2026-07-01T08:35:18.535000+00:00,2026-07-30T17:14:19.184000+00:00,704.65018,0,0.999524,215443469.003188,215443469.003188,16,20,706
trades/binance/BTCUSDT/2026-06.parquet,timestamp_ms,8118466,2026-06-04T00:54:34.149000+00:00,2026-06-30T23:58:25.500000+00:00,647.064264,0,0.900672,68.129207,3162.27766,27,648,649
trades/binance/BTCUSDT/2026-07.parquet,timestamp_ms,6363416,2026-07-01T00:54:13.237000+00:00,2026-07-31T23:56:46.018000+00:00,743.042439,0,0.843782,100.0,2154.43469,31,744,745


depth/binance/BTCUSDT/2026-06.parquet | gaps {'>1s': 649, '>10s': 647, '>60s': 647, '>300s': 647} | largest gaps (s, start) [(3828.0, '2026-06-25T12:55:58.223000+00:00'), (3650.9, '2026-06-22T05:55:07.726000+00:00'), (3494.2, '2026-06-05T15:55:48.826000+00:00'), (3434.9, '2026-06-23T19:54:30.630000+00:00')]
depth/binance/BTCUSDT/2026-07.parquet | gaps {'>1s': 752, '>10s': 743, '>60s': 743, '>300s': 743} | largest gaps (s, start) [(3586.3, '2026-07-05T20:55:12.975000+00:00'), (3509.1, '2026-07-18T10:55:32.533000+00:00'), (3465.5, '2026-07-22T13:55:05.033000+00:00'), (3442.9, '2026-07-20T10:57:22.632000+00:00')]
depth/binance/ETHUSDT/2026-06.parquet | gaps {'>1s': 649, '>10s': 647, '>60s': 647, '>300s': 647} | largest gaps (s, start) [(3828.0, '2026-06-25T12:55:58.224000+00:00'), (3650.9, '2026-06-22T05:55:07.726000+00:00'), (3494.2, '2026-06-05T15:55:48.827000+00:00'), (3434.9, '2026-06-23T19:54:30.630000+00:00')]
depth/binance/ETHUSDT/2026-07.parquet | gaps {'>1s': 751, '>10s': 743, '>

## 4 · Trades: ordering, id continuity, per-minute activity (compared later with the 2024 Layer 1 large-trade definition)

In [ ]:
TRADES = {}; LARGE = {'BTCUSDT': 0.1, 'ETHUSDT': 2.0}
@section('trades')
def _trades():
    for r in SCOPE:
        if r['layout'] != 'trades_like': continue
        roles = ROLES[r['path']]; pt = primary_time(roles); pc_, qc, sc = (roles.get('price') or [None])[0], (roles.get('qty') or [None])[0], (roles.get('side') or [None])[0]; ic = (roles.get('trade_id') or [None])[0]
        if not (pt and pc_ and qc): continue
        path = TG / r['path']; opened(path); pf = pq.ParquetFile(path); st = {}; minute = Counter(); large = Counter(); n = 0; side_ct = Counter(); id_prev = None; id_contig = id_gap = id_dup = 0; pmin, pmax, qmin, qmax = 1e18, -1e18, 1e18, -1e18; sym = r['symbol_from_path']
        for b in pf.iter_batches(batch_size=1_000_000, columns=[c for c in (pt, pc_, qc, sc, ic) if c]):
            t = to_ns(b.column(pt), st); q = np.asarray(b.column(qc).cast(pa.float64()).to_numpy(zero_copy_only=False), float); p = np.asarray(b.column(pc_).cast(pa.float64()).to_numpy(zero_copy_only=False), float); n += len(t)
            mm = pd.Series(t // 60_000_000_000).value_counts()
            for k, v in mm.items(): minute[int(k)] += int(v)
            if sym in LARGE:
                lg = pd.Series((t // 60_000_000_000)[q >= LARGE[sym]]).value_counts()
                for k, v in lg.items(): large[int(k)] += int(v)
            pmin, pmax, qmin, qmax = min(pmin, np.nanmin(p)), max(pmax, np.nanmax(p)), min(qmin, np.nanmin(q)), max(qmax, np.nanmax(q))
            if sc: side_ct.update(pd.Series(b.column(sc).to_pandas()).astype(str).value_counts().to_dict())
            if ic:
                ids = b.column(ic).to_numpy(zero_copy_only=False)
                if ids.dtype.kind in 'iu':
                    ids = ids.astype(np.int64); x = ids if id_prev is None else np.r_[id_prev, ids]; d = np.diff(x); id_contig += int((d == 1).sum()); id_gap += int((d > 1).sum()); id_dup += int((d <= 0).sum()); id_prev = int(ids[-1])
        per_min = np.array(list(minute.values())); span_min = (max(minute) - min(minute) + 1) if minute else 0
        TRADES[r['path']] = {'symbol': sym, 'rows': n, 'side_values': dict(side_ct.most_common(5)), 'price_range': [pmin, pmax], 'qty_range': [qmin, qmax], 'trade_id_col': ic, 'id_contiguous': id_contig, 'id_gaps': id_gap, 'id_dup_or_backwards': id_dup,
                             'minutes_with_trades': len(minute), 'minutes_in_span': span_min, 'trades_per_active_minute_median': float(np.median(per_min)) if len(per_min) else None, 'trades_per_active_minute_p99': float(np.percentile(per_min, 99)) if len(per_min) else None,
                             'large_trades_per_minute_mean_over_span': (sum(large.values()) / span_min) if (sym in LARGE and span_min) else None, 'large_trades_per_active_minute_mean': (sum(large.values()) / len(minute)) if (sym in LARGE and minute) else None, 'large_threshold_used': LARGE.get(sym), 'minute_counts': {int(k): int(v) for k, v in list(minute.items())[:0]}}
        TRADES[r['path']]['_minute'] = minute; TRADES[r['path']]['_large'] = large
    display(pd.DataFrame({k: {x: v[x] for x in v if not x.startswith('_') and x != 'minute_counts'} for k, v in TRADES.items()}).T)
REPORT['trades'] = {k: {x: y for x, y in v.items() if not x.startswith('_')} for k, v in TRADES.items()}

,symbol,rows,side_values,price_range,qty_range,trade_id_col,id_contiguous,id_gaps,id_dup_or_backwards,minutes_with_trades,minutes_in_span,trades_per_active_minute_median,trades_per_active_minute_p99,large_trades_per_minute_mean_over_span,large_trades_per_active_minute_mean,large_threshold_used
trades/binance/BTCUSDT/2026-06.parquet,BTCUSDT,8118466,"{'True': 4145848, 'False': 3972618}","[58380.0, 67287.93]","[1e-05, 33.21694]",trade_id,8117818,647,0,3924,38825,1377.0,11445.98,2.354359,23.294597,0.1
trades/binance/BTCUSDT/2026-07.parquet,BTCUSDT,6363416,"{'True': 3245599, 'False': 3117817}","[58360.25, 66869.79]","[1e-05, 20.48]",trade_id,6362672,743,0,4501,44583,987.0,6935.0,1.950407,19.31904,0.1
trades/binance/ETHUSDT/2026-06.parquet,ETHUSDT,7528634,"{'True': 4004745, 'False': 3523889}","[1521.86, 1849.54]","[0.0001, 224.6556]",trade_id,7527986,647,0,3905,38825,1123.0,12156.76,1.97633,19.649424,2.0
trades/binance/ETHUSDT/2026-07.parquet,ETHUSDT,5145787,"{'True': 2601296, 'False': 2544491}","[1565.68, 1972.3]","[0.0001, 419.0583]",trade_id,5145043,743,0,4495,44583,758.0,7666.96,1.483929,14.718131,2.0


## 5 · Can the data reconstruct a valid order book? (wide-layout check, then snapshot + depth replay with checkpoint comparison)

In [ ]:
def norm_side(arr):
    """+1 bid, -1 ask, 0 unknown. Returns (array, note)."""
    t = arr.type
    if pa.types.is_boolean(t): return np.where(arr.to_numpy(zero_copy_only=False).astype(bool), 1, -1).astype(np.int8), 'boolean (True->bid ASSUMED)'
    if pa.types.is_integer(t) or pa.types.is_floating(t):
        v = np.asarray(arr.cast(pa.float64()).to_numpy(zero_copy_only=False)); u = set(np.unique(v[np.isfinite(v)]).tolist())
        if u <= {-1.0, 1.0}: return np.where(v > 0, 1, -1).astype(np.int8), 'numeric +-1 (+1->bid ASSUMED)'
        return np.where(v == 0, 1, np.where(v == 1, -1, 0)).astype(np.int8), 'numeric 0/1 (0->bid, 1->ask ASSUMED)'
    s = pd.Series(arr.to_pandas()).astype(str).str.lower().str[:1]
    return np.where(s.isin(['b']), 1, np.where(s.isin(['a', 's']), -1, 0)).astype(np.int8), 'string first letter (b->bid, a/s->ask)'
def flat_levels(col):
    """list<list<num|str>> or list<struct> -> (parent_idx, price, qty) float arrays"""
    par = pc.list_parent_indices(col).to_numpy(zero_copy_only=False); inner = pc.list_flatten(col); it = inner.type
    if pa.types.is_struct(it):
        names = [f.name for f in it]; pn = next((n for n in names if re.search(r'pr|px', n, re.I)), names[0]); qn = next((n for n in names if re.search(r'q|sz|size|amount', n, re.I)), names[-1])
        p = pc.struct_field(inner, pn); q = pc.struct_field(inner, qn)
    else: p = pc.list_element(inner, 0); q = pc.list_element(inner, 1)
    return par, np.asarray(p.cast(pa.float64()).to_numpy(zero_copy_only=False)), np.asarray(q.cast(pa.float64()).to_numpy(zero_copy_only=False))
def events_from_batch(tb, roles, layout, tstate, tcol, sidestate):
    """unified depth events: dict(t, side, price, qty, row, U, u)"""
    t = to_ns(tb.column(tcol), tstate); out = {}
    if layout == 'nested':
        parts = []
        for key, sd in (('nested_bids', 1), ('nested_asks', -1)):
            for c in roles.get(key, []):
                par, p, q = flat_levels(tb.column(c)); parts.append((par, np.full(len(par), sd, np.int8), p, q))
        if not parts: raise ValueError('nested layout without bid/ask list columns')
        par = np.concatenate([x[0] for x in parts]); side = np.concatenate([x[1] for x in parts]); p = np.concatenate([x[2] for x in parts]); q = np.concatenate([x[3] for x in parts])
        o = np.argsort(par, kind='stable'); par, side, p, q = par[o], side[o], p[o], q[o]
        out = dict(t=t[par], side=side, price=p, qty=q, row=par)
        if roles.get('final_id'): out['u'] = tb.column(roles['final_id'][0]).to_numpy(zero_copy_only=False).astype(np.int64)[par]
    else:
        sc = roles['side'][0]; side, note = norm_side(tb.column(sc)); sidestate['note'] = note
        out = dict(t=t, side=side, price=np.asarray(tb.column(roles['price'][0]).cast(pa.float64()).to_numpy(zero_copy_only=False)), qty=np.asarray(tb.column(roles['qty'][0]).cast(pa.float64()).to_numpy(zero_copy_only=False)), row=np.arange(len(t)))
        if roles.get('final_id'): out['u'] = tb.column(roles['final_id'][0]).to_numpy(zero_copy_only=False).astype(np.int64)
    return out
def rg_time_bounds(pf, tcol):
    ci = pf.schema_arrow.names.index(tcol); res = []
    for g in range(pf.metadata.num_row_groups):
        s = pf.metadata.row_group(g).column(ci).statistics; res.append((s.min, s.max) if (s is not None and s.has_min_max) else None)
    return res

WIDE = {}
@section('wide-layout-check')
def _wide():
    for r in SCOPE:
        if r['layout'] != 'wide': continue
        path = TG / r['path']; roles = ROLES[r['path']]; opened(path); sch = SCHEMA[r['path']]; cols = {}
        for n in roles['wide']:
            m = re.search(r'(bid|ask|^b|^a)', n, re.I); sd = 'bid' if re.match(r'b', m.group(1), re.I) else 'ask'; kind = 'qty' if re.search(r'qty|size|sz|amount|vol', n, re.I) else 'price'; lv = int(re.findall(r'\d+', n)[-1]); cols.setdefault((sd, kind), {})[lv] = n
        pf = pq.ParquetFile(path); tb = next(pf.iter_batches(batch_size=300_000)); df = tb.to_pandas()
        if not ('bid', 'price') in cols or not ('ask', 'price') in cols: WIDE[r['path']] = {'note': 'could not identify bid/ask price ladders'}; continue
        bl = [cols[('bid', 'price')][k] for k in sorted(cols[('bid', 'price')])]; al = [cols[('ask', 'price')][k] for k in sorted(cols[('ask', 'price')])]
        B = df[bl].to_numpy(float); A = df[al].to_numpy(float); crossed = float((B[:, 0] >= A[:, 0]).mean()); mono_b = float((np.diff(B, axis=1) <= 0).all(axis=1).mean()); mono_a = float((np.diff(A, axis=1) >= 0).all(axis=1).mean())
        mid = (B[:, 0] + A[:, 0]) / 2; spr = (A[:, 0] - B[:, 0]) / mid * 1e4; span = (A[:, -1] - B[:, -1]) / mid * 1e4
        WIDE[r['path']] = {'levels_per_side': len(bl), 'sample_rows': len(df), 'crossed_share': crossed, 'bid_monotone_share': mono_b, 'ask_monotone_share': mono_a, 'spread_bps_median': float(np.nanmedian(spr)), 'visible_span_bps_median': float(np.nanmedian(span)), 'qty_columns_found': ('bid', 'qty') in cols and ('ask', 'qty') in cols}
    print(pd.DataFrame(WIDE).T if WIDE else 'no wide-layout files')
REPORT['wide_book'] = WIDE

SNAPS = {}; REPLAY = {}; EVT = {}
def liq_stats(book, mid):
    o = {}
    for bps in (10, 25):
        lo, hi = mid * (1 - bps / 1e4), mid * (1 + bps / 1e4); o[f'bid{bps}'] = sum(q for p, q in book[1].items() if p >= lo); o[f'ask{bps}'] = sum(q for p, q in book[-1].items() if p <= hi)
    return o

SEGS = {}; SNAPIDS = {}; ANCHORS = {}; SEGSUM = {}
def depth_cols(roles):
    return primary_time(roles), (roles.get('first_id') or [None])[0], (roles.get('final_id') or [None])[0]
@section('segments')
def _segments():
    """message-level segmentation of every in-scope depth file: a segment = run of messages with U == prev_u + 1 (and no time gap > 30 s)"""
    for r in SCOPE:
        if r['category_from_path'] != 'depth_update' or r['layout'] != 'long': continue
        roles = ROLES[r['path']]; tc, fc, lc = depth_cols(roles)
        if not (tc and fc and lc): continue
        path = TG / r['path']; opened(path); pf = pq.ParquetFile(path); st = {}; T_, U_, L_, N_ = [], [], [], []; last = None
        for b in pf.iter_batches(batch_size=2_000_000, columns=[tc, fc, lc]):
            t = to_ns(b.column(tc), st); U = b.column(fc).to_numpy(zero_copy_only=False).astype(np.int64); u = b.column(lc).to_numpy(zero_copy_only=False).astype(np.int64)
            if len(u) == 0: continue
            chg = np.ones(len(u), bool); chg[1:] = u[1:] != u[:-1]; idx = np.flatnonzero(chg); cnt = np.diff(np.r_[idx, len(u)])
            if last is not None and u[0] == last: N_[-1] += int(cnt[0]); idx = idx[1:]; cnt = cnt[1:]
            T_.append(t[idx]); U_.append(U[idx]); L_.append(u[idx]); N_.extend(cnt.tolist()); last = int(u[-1])
        T_ = np.concatenate(T_); U_ = np.concatenate(U_); L_ = np.concatenate(L_); N_ = np.array(N_, np.int64)
        brk = np.ones(len(T_), bool); brk[1:] = (U_[1:] != L_[:-1] + 1) | ((T_[1:] - T_[:-1]) > 30 * 10 ** 9); sid = np.cumsum(brk) - 1; first = np.flatnonzero(brk); last_i = np.r_[first[1:] - 1, len(T_) - 1]
        SEG = pd.DataFrame({'t0': T_[first], 't1': T_[last_i], 'first_U': U_[first], 'last_u': L_[last_i], 'msgs': last_i - first + 1, 'rows': np.add.reduceat(N_, first), 'first_msg_rows': N_[first]}); SEG['dur_s'] = (SEG.t1 - SEG.t0) / 1e9
        SEGS[r['path']] = SEG; dur = SEG.dur_s; span = (SEG.t1.max() - SEG.t0.min()) / 1e9
        SEGSUM[r['path']] = {'segments': len(SEG), 'messages': int(len(T_)), 'duration_s_min_med_max': [float(dur.min()), float(dur.median()), float(dur.max())], 'covered_share_of_span': float(dur.sum() / max(span, 1)), 'msgs_per_segment_median': float(SEG.msgs.median()), 'rows_per_message_median': float(np.median(N_)), 'first_message_rows_median': float(SEG.first_msg_rows.median()), 'first_message_rows_p90': float(SEG.first_msg_rows.quantile(.9)),
                             'start_minute_of_hour_top': pd.Series((pd.to_datetime(SEG.t0, unit='ns', utc=True).dt.minute)).value_counts().head(4).to_dict(), 'gap_between_segments_s_median': float(np.median((SEG.t0.to_numpy()[1:] - SEG.t1.to_numpy()[:-1]) / 1e9)) if len(SEG) > 1 else None,
                             'id_jump_between_segments_median': float(np.median(SEG.first_U.to_numpy()[1:] - SEG.last_u.to_numpy()[:-1])) if len(SEG) > 1 else None}
    print(pd.DataFrame(SEGSUM).T)
    for r in SCOPE:
        if r['category_from_path'] != 'snapshot' or r['layout'] != 'long': continue
        roles = ROLES[r['path']]; tc = primary_time(roles); lc = (roles.get('final_id') or [None])[0]
        if not (tc and lc): continue
        path = TG / r['path']; opened(path); df = pd.read_parquet(path, columns=[tc, lc]).drop_duplicates(lc); st = {}; tn = to_ns(pa.array(df[tc].to_numpy()), st) if not pd.api.types.is_datetime64_any_dtype(df[tc]) else df[tc].astype('int64').to_numpy()
        SNAPIDS.setdefault(r['symbol_from_path'], []).extend([(r['path'], int(a), int(b)) for a, b in zip(tn, df[lc].to_numpy())])
    for sym, lst in SNAPIDS.items():
        rows = []
        for (sp, t, L) in lst:
            hit = None; nxt = None
            for dp, SEG in SEGS.items():
                if sym not in dp: continue
                m = (SEG.first_U.to_numpy() <= L + 1) & (L + 1 <= SEG.last_u.to_numpy())
                if m.any(): hit = (dp, int(np.flatnonzero(m)[0]))
                t_in = (SEG.t0.to_numpy() <= t) & (t <= SEG.t1.to_numpy())
                after = SEG.t0.to_numpy()[SEG.t0.to_numpy() > t]
                if len(after): nxt = min(nxt, (after.min() - t) / 1e9) if nxt is not None else (after.min() - t) / 1e9
            rows.append({'snapshot_file': sp, 't': t, 'L': L, 'anchorable_by_id': hit is not None, 'seg': hit, 'next_segment_start_after_s': nxt})
        ANCHORS[sym] = rows
    A = {s: {'snapshots': len(v), 'anchorable_by_update_id': sum(x['anchorable_by_id'] for x in v), 'median_seconds_to_next_depth_segment': float(np.median([x['next_segment_start_after_s'] for x in v if x['next_segment_start_after_s'] is not None])) if any(x['next_segment_start_after_s'] is not None for x in v) else None} for s, v in ANCHORS.items()}
    SEGSUM['_snapshot_anchoring'] = A; print(json.dumps(A, indent=1, default=jsonable))
REPORT['segments'] = SEGSUM

SNAPS = {}; REPLAY = {}; EVT = {}
def load_snapshot_book(path, tc, lc, L):
    pf = pq.ParquetFile(TG / path); roles = ROLES[path]; sc, pc_, qc = roles['side'][0], roles['price'][0], roles['qty'][0]; out = {1: {}, -1: {}}
    for b in pf.iter_batches(batch_size=500_000, columns=[lc, sc, pc_, qc]):
        m = np.asarray(b.column(lc).to_numpy(zero_copy_only=False)) == L
        if not m.any(): continue
        sd, _ = norm_side(b.column(sc)); p = np.asarray(b.column(pc_).to_numpy(zero_copy_only=False), float); q = np.asarray(b.column(qc).to_numpy(zero_copy_only=False), float)
        for s_, p_, q_ in zip(sd[m], p[m], q[m]):
            if q_ > 0 and s_ != 0: out[int(s_)][float(p_)] = float(q_)
    return out
def prune(bk, side, K):
    if len(bk) > K + 50:
        ks = sorted(bk, reverse=(side == 1))
        for p in ks[K:]: del bk[p]
def replay_symbol(sym, row_cap=2_000_000, win_s=600, K=1000):
    ds = sorted([r for r in SCOPE if r['symbol_from_path'] == sym and r['category_from_path'] == 'depth_update' and r['layout'] == 'long' and r['path'] in SEGS], key=lambda r: r['path'])
    tr = next((r for r in SCOPE if r['symbol_from_path'] == sym and r['layout'] == 'trades_like'), None)
    info = {'depth_file': None, 'trades_file': tr and tr['path'], 'status': 'SKIPPED: no segmented depth file'}
    if not ds: return info
    anch = [x for x in ANCHORS.get(sym, []) if x['anchorable_by_id']]; info['anchorable_snapshots'] = len(anch); info['snapshots_total'] = len(ANCHORS.get(sym, []))
    dense = any(SEGSUM[r['path']]['first_message_rows_median'] >= 200 for r in ds)
    if anch:
        anch = sorted(anch, key=lambda x: -(SEGS[x['seg'][0]].loc[x['seg'][1]].last_u - x['L'])); mode = 'snapshot'; dp, si = anch[0]['seg']; dfile = next(r for r in ds if r['path'] == dp); init = anch[0]
    else:
        dfile = ds[0]; SEG = SEGS[dfile['path']]; ok = SEG[(SEG.msgs >= 1000)]; si = int(ok.index[len(ok) // 2]) if len(ok) else int(SEG.msgs.idxmax()); mode = 'dense_first_message' if dense else 'unanchored'; init = None
    SEG = SEGS[dfile['path']]; seg = SEG.loc[si]; info.update({'depth_file': dfile['path'], 'anchor_mode': mode, 'segment_index': int(si), 'segment_start': pd.Timestamp(int(seg.t0), unit='ns', tz='UTC').isoformat(), 'segment_msgs': int(seg.msgs), 'segment_duration_s': float(seg.dur_s)})
    dr = ROLES[dfile['path']]; tcol, fcol, lcol = depth_cols(dr); pf = pq.ParquetFile(TG / dfile['path']); opened(TG / dfile['path']); bounds = rg_time_bounds(pf, tcol); st = {}; ss = {}; chunks = []; rows = 0
    t_hi = (init['t'] if init else int(seg.t0)) + win_s * 10 ** 9; cols = [c for c in dict.fromkeys([tcol, fcol, lcol] + dr['side'][:1] + dr['price'][:1] + dr['qty'][:1])]
    u_lo = (init['L'] if init else int(seg.first_U) - 1)
    for g in range(pf.metadata.num_row_groups):
        tb = pf.read_row_group(g, columns=cols); ev = events_from_batch(tb, dr, 'long', st, tcol, ss); ev['U'] = tb.column(fcol).to_numpy(zero_copy_only=False).astype(np.int64)
        m = (ev['u'] > u_lo) & (ev['u'] <= int(seg.last_u)) & (ev['t'] <= t_hi)
        if m.any(): chunks.append({k: v[m] for k, v in ev.items()}); rows += int(m.sum())
        if rows >= row_cap: break
    info['depth_side_encoding'] = ss.get('note')
    if not chunks: info['status'] = 'NO DEPTH EVENTS found for the selected segment'; return info
    E = {k: np.concatenate([c[k] for c in chunks]) for k in chunks[0]}; n = len(E['t']); info['events_replayed'] = n
    info['qty_negative_share'] = float((E['qty'] < 0).mean()); info['qty_zero_share'] = float((E['qty'] == 0).mean())
    book = {1: {}, -1: {}}; seen = {1: set(), -1: set()}; u_arr = E['u']; first_u = int(u_arr[0])
    if mode == 'snapshot': book = load_snapshot_book(init['snapshot_file'], None, ROLES[init['snapshot_file']]['final_id'][0], init['L']); info['init_snapshot_levels'] = [len(book[1]), len(book[-1])]
    init_n = 0
    if mode == 'dense_first_message':
        init_n = int((u_arr == first_u).sum())
        for i in range(init_n):
            s_ = int(E['side'][i]);
            if s_ and E['qty'][i] > 0: book[s_][float(E['price'][i])] = float(E['qty'][i])
        info['init_rows_from_first_message'] = init_n
    for s_ in (1, -1): seen[s_].update(book[s_].keys())
    dl = np.zeros(n); newq = np.zeros(n); known = np.zeros(n, bool); cps = []; crossed = 0; ncp = 0; next_cp = E['t'][init_n]; phantom = 0; removals = 0; T = E['t']; SD = E['side']; PR = E['price']; QT = E['qty']
    warm = int(seg.t0) + (0 if mode != 'unanchored' else 120 * 10 ** 9)
    for i in range(init_n, n):
        s_ = int(SD[i]); p_ = float(PR[i]); q_ = float(QT[i])
        if s_ == 0: continue
        bk = book[s_]; old = bk.get(p_)
        if q_ <= 1e-12:
            removals += 1
            if old is None: phantom += 1
            bk.pop(p_, None); nw = 0.0
        else: bk[p_] = q_; nw = q_
        kn = (old is not None) or (p_ in seen[s_]) or mode != 'unanchored'
        seen[s_].add(p_); dl[i] = (nw - (old or 0.0)) if kn else 0.0; newq[i] = nw; known[i] = kn
        if len(bk) > K + 50: prune(bk, s_, K)
        ti_ = T[i]
        if ti_ >= next_cp and ti_ >= warm and book[1] and book[-1]:
            bb = max(book[1]); ba = min(book[-1]); ncp += 1; crossed += int(bb >= ba); mid = (bb + ba) / 2; ls = liq_stats(book, mid); cps.append((ti_, mid, (ba - bb) / mid * 1e4, len(book[1]), len(book[-1]), (mid - min(book[1])) / mid * 1e4, (max(book[-1]) - mid) / mid * 1e4, ls['bid10'] + ls['ask10'], ls['bid25'] + ls['ask25'])); next_cp = ti_ + 10 ** 9
    info['status'] = 'OK'; info['book_samples'] = ncp; info['crossed_book_share_of_samples'] = crossed / max(ncp, 1); info['removals'] = removals; info['phantom_removal_rate_(removal_of_level_not_in_book)'] = phantom / max(removals, 1)
    info['delta_known_share_of_events'] = float(known[init_n:].mean()) if n > init_n else None
    CP = pd.DataFrame(cps, columns=['t', 'mid', 'spread_bps', 'n_bid_levels', 'n_ask_levels', 'bid_span_bps', 'ask_span_bps', 'depth10', 'depth25']) if cps else pd.DataFrame()
    if len(CP): info['spread_bps_median'] = float(CP.spread_bps.median()); info['levels_per_side_median'] = [float(CP.n_bid_levels.median()), float(CP.n_ask_levels.median())]; info['visible_span_bps_median'] = [float(CP.bid_span_bps.median()), float(CP.ask_span_bps.median())]; info['depth25_median_base_units'] = float(CP.depth25.median())
    # later in-segment snapshots (independent checkpoints), only if they exist
    cmp_ = []
    for x in ANCHORS.get(sym, []):
        if x['seg'] and x['seg'][0] == dfile['path'] and x['seg'][1] == si and x is not init and x['L'] > u_lo and x['L'] <= u_arr[-1]: cmp_.append(x)
    info['independent_checkpoint_snapshots_in_segment'] = len(cmp_)
    dt = np.diff(T) / 1e6; dt = dt[dt > 0]; info['inter_event_ms_median'] = float(np.median(dt)) if len(dt) else None; info['events_per_second'] = n / max((T[-1] - T[0]) / 1e9, 1e-9)
    EVT[sym] = {'t': T[init_n:], 'side': SD[init_n:], 'price': PR[init_n:], 'delta': dl[init_n:], 'newq': newq[init_n:], 'known': known[init_n:], 'CP': CP, 'trades_file': tr and tr['path'], 'mode': mode}
    return info
@section('replay')
def _replay():
    for sym in ('BTCUSDT', 'ETHUSDT'):
        try: REPLAY[sym] = replay_symbol(sym)
        except Exception as e:
            import traceback; REPLAY[sym] = {'status': 'ERROR: ' + type(e).__name__ + ': ' + str(e)[:200]}; SECTION_ERRORS['replay_' + sym] = traceback.format_exc()[-1200:]
        print(sym, json.dumps(REPLAY[sym], indent=1, default=jsonable)[:2500])
REPORT['replay'] = REPLAY; REPORT['wide_book'] = WIDE

no wide-layout files
                                      segments messages       duration_s_min_med_max covered_share_of_span msgs_per_segment_median rows_per_message_median first_message_rows_median first_message_rows_p90  \
depth/binance/BTCUSDT/2026-06.parquet      648  1966112    [4.618, 299.942, 299.999]              0.083202                  3040.0                    15.0                      15.0                   73.3   
depth/binance/BTCUSDT/2026-07.parquet      744  2259460  [273.906, 299.945, 300.406]              0.083412                  3036.0                     9.0                       9.0                   44.0   
depth/binance/ETHUSDT/2026-06.parquet      648  1965315    [4.617, 299.941, 299.997]              0.083201                  3040.0                     9.0                       9.0                   63.3   
depth/binance/ETHUSDT/2026-07.parquet      744  2253045  [273.907, 299.945, 300.406]              0.083411                  3035.0                     

## 6 · Order-level vs price-level information, cancel-vs-execute separability, heuristic spoof/layering/withdrawal prevalence (descriptive only, not labels)

In [ ]:
ORDERLVL = {}; SPOOF = {}
@section('order-level-availability')
def _ol():
    for r in SCOPE:
        roles = ROLES[r['path']]; sch = SCHEMA[r['path']]; evcol = roles.get('event_type', []); oid = roles.get('order_id', []); vals = None
        if evcol:
            try: vals = pd.Series(next(pq.ParquetFile(TG / r['path']).iter_batches(batch_size=200_000, columns=[evcol[0]])).column(0).to_pandas()).astype(str).value_counts().head(8).to_dict()
            except Exception: vals = None
        ORDERLVL[r['path']] = {'layout': r['layout'], 'order_id_columns': oid, 'event_type_columns': evcol, 'event_type_sample_values': vals, 'level_columns': {k: v for k, v in roles.items() if k in ('price', 'qty', 'side', 'nested_bids', 'nested_asks', 'wide')}}
    anyoid = any(v['order_id_columns'] for v in ORDERLVL.values() if v['layout'] != 'trades_like'); anyev = any(v['event_type_columns'] for v in ORDERLVL.values() if v['layout'] != 'trades_like')
    ORDERLVL['_summary'] = {'order_ids_in_book_files': anyoid, 'event_type_columns_in_book_files': anyev, 'trade_files_with_ids': [k for k, v in TRADES.items() if v.get('trade_id_col')], 'conclusion': ('order-level (L3) information PRESENT' if anyoid else 'ONLY aggregated price-level information; no per-order identity') + ('; explicit event-type column present (inspect values)' if anyev else '; no explicit add/cancel/execute type column: add vs reduce is inferred from quantity change')}
    print(json.dumps(ORDERLVL['_summary'], indent=1))
REPORT['order_level'] = ORDERLVL

@section('microstructure-heuristics')
def _heur():
    for sym, E in EVT.items():
        T, SD, PR, DL, NQ, CP = E['t'], E['side'], E['price'], E['delta'], E['newq'], E['CP']; n = len(T); out = {'events': n, 'anchor_mode': E.get('mode'), 'known_delta_share': float(E['known'].mean()) if 'known' in E else None, 'window_hours': float((T[-1] - T[0]) / 3.6e12)}
        # -- cancel vs execute: reductions explained by trades at the same price within +-1 bucket of 100 ms
        trade_note = None; vol_by = {}
        if E['trades_file']:
            tr = next(r for r in SCOPE if r['path'] == E['trades_file']); trr = ROLES[tr['path']]; pt = primary_time(trr); pf = pq.ParquetFile(TG / tr['path']); opened(TG / tr['path']); st = {}; parts = []; bnd = rg_time_bounds(pf, pt)
            for g in range(pf.metadata.num_row_groups):
                tb = pf.read_row_group(g, columns=[pt, trr['price'][0], trr['qty'][0]]); tt = to_ns(tb.column(0), st); m = (tt >= T[0] - 10 ** 9) & (tt <= T[-1] + 10 ** 9)
                if m.any(): parts.append(pd.DataFrame({'t': tt[m], 'p': np.asarray(tb.column(1).cast(pa.float64()).to_numpy(zero_copy_only=False))[m], 'q': np.asarray(tb.column(2).cast(pa.float64()).to_numpy(zero_copy_only=False))[m]}))
            TRW = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame(columns=['t', 'p', 'q'])
        else: TRW = pd.DataFrame(columns=['t', 'p', 'q'])
        out['trades_in_window'] = len(TRW)
        red = DL < 0
        if red.any() and len(TRW):
            BK = 100_000_000; ev = pd.DataFrame({'p': PR[red].round(8), 'b': T[red] // BK, 'r': -DL[red]}).groupby(['p', 'b']).r.sum().reset_index(); tv = TRW.assign(p=TRW.p.round(8), b=TRW.t // BK).groupby(['p', 'b']).q.sum()
            vol = np.zeros(len(ev))
            for sh in (-1, 0, 1): vol += tv.reindex(pd.MultiIndex.from_arrays([ev.p, ev.b + sh])).fillna(0).to_numpy()
            ex = np.minimum(ev.r.to_numpy(), vol); out['reduction_volume_total'] = float(ev.r.sum()); out['reduction_explained_by_trades_share'] = float(ex.sum() / ev.r.sum()); out['reduction_unexplained_share_(cancel-like_or_timing)'] = float(1 - ex.sum() / ev.r.sum())
            out['cancel_vs_execute_note'] = 'attribution by price + 100 ms buckets (+-1): approximate; trades and depth feeds may be stamped by different clocks'
        # -- spoof-like candidates
        add = DL > 0
        if add.sum() > 1000:
            thr = float(np.quantile(DL[add], 0.999)); cand = np.flatnonzero(add & (DL >= thr)); out['big_add_threshold_q999'] = thr; out['big_adds'] = int(len(cand))
            o = np.lexsort((T, PR, SD)); pos = np.empty(n, np.int64); pos[o] = np.arange(n); Ts, Ps, Ss, Ns = T[o], PR[o], SD[o], NQ[o]
            if len(cand) > 20000: cand = cand[np.linspace(0, len(cand) - 1, 20000).astype(int)]; out['candidates_subsampled_to'] = 20000
            rows = []
            trs = TRW.sort_values('t'); tp = {}
            for c in cand:
                k = pos[c]; q0 = NQ[c]; t0_ = T[c]; vanish = None; j = k + 1
                while j < n and Ps[j] == Ps[k] and Ss[j] == Ss[k] and Ts[j] - t0_ <= 10 * 10 ** 9:
                    if Ns[j] <= 0.2 * q0: vanish = Ts[j]; break
                    j += 1
                if vanish is None: rows.append((t0_, SD[c], PR[c], q0, np.nan, np.nan)); continue
                lo, hi = np.searchsorted(trs.t.to_numpy(), [t0_, vanish]); seg = trs.iloc[lo:hi]; exv = float(seg.q[np.isclose(seg.p, PR[c], rtol=0, atol=1e-9)].sum()) if len(seg) else 0.0
                rows.append((t0_, SD[c], PR[c], q0, (vanish - t0_) / 1e9, exv / q0))
            C = pd.DataFrame(rows, columns=['t', 'side', 'price', 'qty', 'ttv_s', 'exec_frac']); van = C.ttv_s.notna(); sp = van & (C.exec_frac <= 0.1)
            out['vanish_within_10s_share'] = float(van.mean()); out['vanish_with_exec<=10pct_share_(spoof-like_pattern)'] = float(sp.mean()); out['spoof_like_count'] = int(sp.sum()); out['spoof_like_per_hour_(on_candidates_examined)'] = float(sp.sum() / max(out['window_hours'], 1e-9))
            out['ttv_s_median_spoof_like'] = float(C.ttv_s[sp].median()) if sp.any() else None
            if len(CP) and sp.any():
                mid = CP.mid.to_numpy()[np.clip(np.searchsorted(CP.t.to_numpy(), C.t[sp].to_numpy()), 0, len(CP) - 1)]; out['spoof_like_distance_from_mid_bps_median_approx'] = float(np.median(np.abs(C.price[sp].to_numpy() - mid) / mid * 1e4))
            if sp.sum() >= 3:
                S = C[sp].assign(sec=lambda d: (d.t // 10 ** 9)); L = S.groupby(['side', 'sec']).price.nunique(); out['layering_like_clusters_(>=3_levels_same_side_same_second)'] = int((L >= 3).sum())
            out['examples_spoof_like'] = [{'t': pd.Timestamp(int(a), unit='ns', tz='UTC').isoformat(), 'side': 'bid' if b > 0 else 'ask', 'price': float(c), 'qty': float(d), 'vanish_s': float(e), 'exec_frac': float(f)} for a, b, c, d, e, f in C[sp].head(3).itertuples(index=False)]
        # -- liquidity-withdrawal-like episodes
        if len(CP) > 120 and E.get('mode') != 'unanchored':
            d25 = CP.depth25.to_numpy(); t = CP.t.to_numpy(); mid = CP.mid.to_numpy(); hit = np.zeros(len(CP), bool); win = 60
            for i in range(len(CP) - 1):
                j = np.searchsorted(t, t[i] + win * 10 ** 9)
                if j <= i + 1: continue
                if d25[i] > 0 and d25[i + 1:j].min() < 0.5 * d25[i] and abs(mid[i + 1:j] - mid[i]).max() / mid[i] * 1e4 < 10: hit[i] = True
            ep = int(((hit[1:] & ~hit[:-1]).sum()) + hit[0]); out['liquidity_withdrawal_like_episodes_(depth25 falls>50pct in 60s, |mid move|<10bps)'] = ep; out['liquidity_withdrawal_like_per_hour'] = ep / max(out['window_hours'], 1e-9)
            out['depth25_series'] = {'samples': len(CP), 'median': float(np.median(d25)), 'q05': float(np.quantile(d25, 0.05)), 'q95': float(np.quantile(d25, 0.95))}
        SPOOF[sym] = out; print(sym, json.dumps(out, indent=1, default=jsonable)[:3000])
REPORT['heuristics'] = SPOOF

{
 "order_ids_in_book_files": false,
 "event_type_columns_in_book_files": false,
 "trade_files_with_ids": [
  "trades/binance/BTCUSDT/2026-06.parquet",
  "trades/binance/BTCUSDT/2026-07.parquet",
  "trades/binance/ETHUSDT/2026-06.parquet",
  "trades/binance/ETHUSDT/2026-07.parquet"
 ],
 "conclusion": "ONLY aggregated price-level information; no per-order identity; no explicit add/cancel/execute type column: add vs reduce is inferred from quantity change"
}
BTCUSDT {
 "events": 30812,
 "anchor_mode": "snapshot",
 "known_delta_share": 1.0,
 "window_hours": 0.08340861111111111,
 "trades_in_window": 0,
 "big_add_threshold_q999": 111.19600984000165,
 "big_adds": 13,
 "vanish_within_10s_share": 0.0,
 "vanish_with_exec<=10pct_share_(spoof-like_pattern)": 0.0,
 "spoof_like_count": 0,
 "spoof_like_per_hour_(on_candidates_examined)": 0.0,
 "ttv_s_median_spoof_like": null,
 "examples_spoof_like": [],
 "liquidity_withdrawal_like_episodes_(depth25 falls>50pct in 60s, |mid move|<10bps)": 0,
 "liquid

## 7 · Verified labels? Compatibility with the 2024 Layer 1 data

In [ ]:
LABELS = {}; COMPAT = {}
@section('labels')
def _lab():
    cols = {k: v.get('label_like', []) for k, v in ROLES.items() if v.get('label_like')}
    kw = r'ground.?truth|manipulat|spoof|layering|wash.?trad|pump|label|annotat|regulator|enforcement|sec |cftc|verified'
    mentions = [(p, ln.strip()[:200]) for p, tx in PROV['text_files'].items() for ln in tx.splitlines() if re.search(kw, ln, re.I)][:12]
    LABELS.update({'label_like_columns': cols, 'readme_mentions': mentions, 'custom_parquet_metadata_keys': PROV.get('parquet_custom_metadata_keys', {}),
                   'verdict': 'NO independently verified manipulation labels found in columns or shipped text' if not cols and not mentions else 'POSSIBLE label-like columns / text found: INSPECT MANUALLY (column names are not proof of independent verification)'})
    print(json.dumps(LABELS, indent=1, default=jsonable)[:2500])
REPORT['labels'] = LABELS

@section('layer1-compat')
def _compat():
    if not L1.is_dir(): COMPAT['note'] = 'Layer 1 folder not found'; return
    for sym, low in (('BTCUSDT', 'btcusdt'), ('ETHUSDT', 'ethusdt')):
        fs = sorted((L1 / 'partitions').glob(f'{low}_*split*.parquet'));
        if not fs: continue
        df = pd.concat([pd.read_parquet(f) for f in fs], ignore_index=True); dt = pd.to_datetime(df['datetime'], utc=True)
        c = {'l1_range': [dt.min().isoformat(), dt.max().isoformat()], 'l1_bars': len(df), 'l1_bar_seconds': float(dt.diff().dt.total_seconds().median()), 'l1_price_range': [float(df['close'].min()), float(df['close'].max())],
             'l1_trades_count_median': float(df['trades_count'].median()), 'l1_volume_median': float(df['volume'].median()), 'l1_large_trade_count_mean': float(df['large_trade_count'].mean()), 'l1_large_threshold': LARGE[sym]}
        tr = [v for v in TRADES.values() if v['symbol'] == sym]
        if tr:
            v = tr[0]; tm = TIMESUM.get(next(k for k, x in TRADES.items() if x['symbol'] == sym), {}); c.update({'tg_range': [tm.get('tmin'), tm.get('tmax')], 'tg_trades_per_active_minute_median': v['trades_per_active_minute_median'], 'tg_large_trades_per_minute_mean_over_span': v['large_trades_per_minute_mean_over_span'], 'tg_large_trades_per_active_minute_mean': v.get('large_trades_per_active_minute_mean'), 'tg_price_range': v['price_range'],
                      'time_overlap_with_l1': bool(tm.get('tmin') and tm.get('tmax') and not (pd.Timestamp(tm['tmax']) < dt.min() or pd.Timestamp(tm['tmin']) > dt.max()))})
        COMPAT[sym] = c
    COMPAT['structural_differences'] = ['2024 Layer 1 = 1-minute OHLCV-derived feature bars (6 features); 2026 TGNN data = event-level order book (+trades), no feature bars.', 'No calendar overlap -> no event-by-event linking; models cannot be trained on one and evaluated on the other without a regime/period shift.',
        'Absolute price/volume levels differ across years; any shared representation must be scale-free (relative depth, bps distance, rank levels).', 'Layer 1 trades_count/large_trade_count definitions (thresholds BTC 0.1, ETH 2.0 units) can be recomputed from 2026 trades but trade definitions (raw vs aggregated) must be confirmed.', 'Layer 3 synthetic scenarios live at 1-minute aggregate level; they cannot be applied to the event-level book without a new injection design.']
    print(json.dumps(COMPAT, indent=1, default=jsonable)[:3500])
REPORT['layer1_compat'] = COMPAT

{
 "label_like_columns": {},
 "readme_mentions": [],
 "custom_parquet_metadata_keys": {},
 "verdict": "NO independently verified manipulation labels found in columns or shipped text"
}
{
 "BTCUSDT": {
  "l1_range": [
   "2024-05-01T00:00:00+00:00",
   "2024-07-29T23:59:00+00:00"
  ],
  "l1_bars": 129600,
  "l1_bar_seconds": 60.0,
  "l1_price_range": [
   53778.14,
   71959.67
  ],
  "l1_trades_count_median": 641.0,
  "l1_volume_median": 10.58619,
  "l1_large_trade_count_mean": 46.53976851851852,
  "l1_large_threshold": 0.1,
  "tg_range": [
   "2026-06-04T00:54:34.149000+00:00",
   "2026-06-30T23:58:25.500000+00:00"
  ],
  "tg_trades_per_active_minute_median": 1377.0,
  "tg_large_trades_per_minute_mean_over_span": 2.3543593045717963,
  "tg_large_trades_per_active_minute_mean": 23.294597349643222,
  "tg_price_range": [
   58380.0,
   67287.93
  ],
  "time_overlap_with_l1": false
 },
 "ETHUSDT": {
  "l1_range": [
   "2024-05-01T00:00:00+00:00",
   "2024-07-29T23:59:00+00:00"
  ],
  "l1_ba

## 8 · Feasible causal graph design (within-market microstructure only), sufficiency verdict

In [ ]:
def gv(d, *ks, default=None):
    for k in ks:
        if not isinstance(d, dict) or k not in d: return default
        d = d[k]
    return d
VER = {}
@section('verdict')
def _verdict():
    rep_ok = {}; crit = {}
    for sym in ('BTCUSDT', 'ETHUSDT'):
        R = REPLAY.get(sym, {}); st = R.get('status', 'MISSING'); mode = R.get('anchor_mode')
        if st != 'OK': rep_ok[sym] = 'UNVERIFIED' if (str(st).startswith('SKIPPED') or st == 'MISSING') else 'FAILED'
        elif mode in ('snapshot', 'dense_first_message'):
            rep_ok[sym] = 'VALIDATED' if (R.get('crossed_book_share_of_samples', 1) == 0 and R.get('phantom_removal_rate_(removal_of_level_not_in_book)', 1) < 0.01) else 'FAILED (anchored replay internally inconsistent)'
        else: rep_ok[sym] = 'NOT RECONSTRUCTABLE (no valid anchor: segment start is not a snapshot)'
    crit['anchoring'] = {s: {'mode': REPLAY.get(s, {}).get('anchor_mode'), 'anchorable_snapshots': REPLAY.get(s, {}).get('anchorable_snapshots'), 'snapshots_total': REPLAY.get(s, {}).get('snapshots_total'), 'independent_checkpoints': REPLAY.get(s, {}).get('independent_checkpoint_snapshots_in_segment')} for s in REPLAY}
    crit['depth_coverage'] = {k: {'segments': v.get('segments'), 'covered_share_of_span': v.get('covered_share_of_span'), 'median_duration_s': (v.get('duration_s_min_med_max') or [None, None])[1]} for k, v in SEGSUM.items() if not k.startswith('_')}
    if any('wide' == v for v in LAYOUT.values()) and WIDE and all(w.get('crossed_share', 1) == 0 and w.get('bid_monotone_share', 0) > 0.999 for w in WIDE.values() if 'crossed_share' in w): crit['wide_book_valid'] = True
    if crit.get('wide_book_valid'): rep_ok = {k: ('INTERNALLY CONSISTENT, NOT CHECKPOINT-VERIFIED' if v == 'UNVERIFIED' else v) for k, v in rep_ok.items()}
    crit['reconstruction'] = rep_ok
    inv = max([gv(v, 'inversion_share', default=0) or 0 for v in TIMESUM.values()] + [0]); crit['max_inversion_share'] = inv
    crit['gaps_over_60s'] = {k: gv(v, 'gaps', '>60s') for k, v in TIMESUM.items()}
    crit['span_hours'] = {k: gv(v, 'span_hours') for k, v in TIMESUM.items()}
    crit['trade_side_present'] = all(bool(v.get('side_values')) for v in TRADES.values()) if TRADES else False
    crit['order_ids'] = bool(gv(ORDERLVL, '_summary', 'order_ids_in_book_files'))
    crit['verified_labels'] = not str(LABELS.get('verdict', '')).startswith('NO ')
    crit['levels_per_side_median'] = {s: gv(REPLAY, s, 'levels_per_side_median') for s in REPLAY}
    crit['inter_event_ms_median'] = {s: gv(REPLAY, s, 'inter_event_ms_median') for s in REPLAY}
    VER['criteria'] = crit; ok_rec = all(v in ('VALIDATED', 'INTERNALLY CONSISTENT, NOT CHECKPOINT-VERIFIED') for v in rep_ok.values()) and bool(rep_ok)
    diff_known = [gv(SPOOF, s, 'known_delta_share') for s in SPOOF]; diff_ok = bool(diff_known) and all((x or 0) > 0.8 for x in diff_known) and crit['trade_side_present']
    VER['can_support'] = {'unsupervised_or_self_supervised_book_anomaly_modelling': ok_rec, 'synthetic_injection_experiment_(spoof/layer/withdraw-like)': ok_rec, 'supervised_detection_of_real_manipulation': crit['verified_labels'], 'order_level_spoofing_forensics': crit['order_ids'], 'diff_stream_event_graph_without_full_book_(touched-level state, short windows)': diff_ok}
    unc = []
    if any(v != 'VALIDATED' for v in rep_ok.values()): unc.append('Order-book reconstruction is not fully checkpoint-validated for every symbol (see section 5): replayed books may not equal the exchange book.')
    if not crit['order_ids']: unc.append('No per-order identity: queue position, order lifetime, and who cancelled are unobservable; only net price-level quantity changes are seen.')
    if any(str(v).startswith('NOT RECONSTRUCTABLE') for v in rep_ok.values()): unc.append('The depth files are diff windows whose start is not a snapshot; the book outside levels touched inside a window is unknown. Windows are separated by update-ID discontinuities and cannot be chained.')
    unc.append('Add/cancel/execute are inferred (quantity up / down / down with matching trade); this attribution is approximate and clock-skew sensitive.')
    unc.append('Spoof-like / layering-like / withdrawal-like counts are heuristic prevalence of patterns that also occur in legitimate market making; they are not labels and not evidence of intent.')
    unc.append('Only a short replay window per symbol was examined (<=90 min); statistics are not representative of the whole June-July 2026 span.')
    if not crit['verified_labels']: unc.append('No independent labels: any TGNN evaluation needs synthetic injection into real books or an unsupervised/self-supervised protocol with explicit limits.')
    unc.append('Provenance/licence: confirm on the dataset card; the shipped files may carry no licence text (see section 2). A similarly named dataset exists under another owner; do not conflate.')
    VER['uncertainties'] = unc
    anyna = any(str(v).startswith('NOT RECONSTRUCTABLE') for v in rep_ok.values())
    VER['recommendation'] = ('PROCEED to a TGNN PROTOCOL DESIGN (not training) as a *within-market, synthetic-injection + unsupervised* feasibility study, ONLY after you approve this audit; mandatory non-graph baselines (same inputs, no edges); claims limited to detection of injected/heuristic patterns in reconstructed price-level books.' if ok_rec
                            else ('FULL-BOOK TGNN NOT SUPPORTED by the published files (no valid snapshot anchor for the diff windows). Options for approval: (A) restrict to a DIFF-STREAM event graph inside each window (levels touched so far + trades; honest about partial state), with synthetic injection and non-graph baselines; (B) obtain a continuous capture with anchoring snapshots (e.g. the crypto-lob-stream collector or Binance public data) before any book-state TGNN; (C) drop TGNN from the claims and keep Components 1-3 + fusion. Do NOT train until you choose.' if anyna else 'DO NOT design the TGNN protocol yet: reconstruction failed/unverified (see section 5 and SECTION ERRORS), then re-run the audit.'))
    print(json.dumps(VER, indent=1, default=jsonable)[:3500])
REPORT['verdict'] = VER

GRAPH = {
 'scope': 'Per-symbol graphs (BTCUSDT and ETHUSDT separately). NO cross-symbol edges (no cross-market claims in this audit).',
 'time': 'Event time of the depth feed. Every edge points from an earlier or equal event time to a later one (strictly causal): a node/edge may only use events with time <= t.',
 'snapshot_unit': 'Rolling windows of W seconds (candidate W = 5, 15, 60 s) sampled at event time; one graph per decision time t.',
 'nodes': {'price_level': 'one node per visible price level at time t, indexed by RELATIVE level (best bid = B1.., best ask = A1.. up to K levels per side; K chosen from measured levels_per_side). Features: log(1+qty), signed distance from mid in bps, dqty over last 1/5/30 s, level age since last change, executed volume at the price over the last 1/5 s.',
           'trade_event': 'aggregated trade bursts (per 100 ms) with signed volume, price vs mid (bps)', 'global': 'mid, spread bps, imbalance over top-K, depth within 10/25 bps (readout, not a node)'},
 'edges': ['price adjacency (level i <-> level i+1 on the same side)', 'best-bid <-> best-ask link (spread edge)', 'temporal same-level edge (level state at t-1 -> t, carries the update history; TGNN memory)', 'trade -> level edge (trade executed at/near that level, time <= t)', 'optional: same-side level groups changing within the same time bucket (layering-like co-movement), derived from the data not labelled'],
 'targets_(later_protocol_not_now)': 'self-supervised next-window prediction of depth/imbalance; injected spoof-like (large add that vanishes unexecuted), layering-like (>=3 levels), withdrawal-like (depth within 25 bps falls) perturbations applied to a COPY of the reconstructed book with an explicit mask; held-out time blocks for train/val/test.',
 'must_have_baselines': ['non-graph model on the same window features (GRU/Transformer/XGBoost)', 'rolling-z-score depth/imbalance anomaly baseline', 'shuffled-edge graph ablation'],
 'not_representable': 'order identity, queue priority, participant identity, cross-venue effects, intent.'}
REPORT['graph_design'] = GRAPH
print(json.dumps(GRAPH, indent=1)[:1200])

{
 "criteria": {
  "anchoring": {
   "BTCUSDT": {
    "mode": "snapshot",
    "anchorable_snapshots": 1,
    "snapshots_total": 38,
    "independent_checkpoints": 0
   },
   "ETHUSDT": {
    "mode": "snapshot",
    "anchorable_snapshots": 1,
    "snapshots_total": 37,
    "independent_checkpoints": 0
   }
  },
  "depth_coverage": {
   "depth/binance/BTCUSDT/2026-06.parquet": {
    "segments": 648,
    "covered_share_of_span": 0.08320158110020617,
    "median_duration_s": 299.942
   },
   "depth/binance/BTCUSDT/2026-07.parquet": {
    "segments": 744,
    "covered_share_of_span": 0.08341172477957251,
    "median_duration_s": 299.945
   },
   "depth/binance/ETHUSDT/2026-06.parquet": {
    "segments": 648,
    "covered_share_of_span": 0.08320126725328114,
    "median_duration_s": 299.941
   },
   "depth/binance/ETHUSDT/2026-07.parquet": {
    "segments": 744,
    "covered_share_of_span": 0.08341138006885997,
    "median_duration_s": 299.945
   }
  },
  "reconstruction": {
   "BTCUSDT": "F

## 9 · Audit tests, persistence, final read-only proof, FINAL AUDIT SHARE BLOCK

In [ ]:
TESTS = []
def tcase(name, fn):
    try: ok, msg = fn()
    except Exception as e: ok, msg = False, f'EXC {type(e).__name__}: {str(e)[:160]}'
    TESTS.append((name, bool(ok), msg)); print(('PASS ' if ok else 'FAIL ') + name + ' | ' + str(msg))
def t_a01():
    now = {str(p.relative_to(TG)): (p.stat().st_size, p.stat().st_mtime_ns) for p in TG.rglob('*') if p.is_file()}; was = {k: (v['size'], v['mtime_ns']) for k, v in PIN_TG.items()}
    if now != was: return False, 'file list/size/mtime changed'
    bad = [k for k in PIN_TG if sha256_file(TG / k) != PIN_TG[k]['sha256']]; return (not bad), f'{len(PIN_TG)} files re-hashed identical' if not bad else f'CHANGED: {bad}'
def t_a02(): bad = [o for o in OPENED if re.search(r'sol', o, re.I) and 'solusdt' in o.lower()]; return (not bad), f'{len(OPENED)} open events, none out-of-scope' if not bad else bad
def t_a03(): s = json.dumps(REPORT, default=jsonable); json.loads(s); return True, f'report serialises ({len(s)//1024} KiB)'
def t_a04():
    bad = []
    for k, v in TIMESUM.items():
        if v.get('rows') is not None and v['rows'] != META[k]['rows']: bad.append((k, v['rows'], META[k]['rows']))
    for k, v in TRADES.items():
        if v['rows'] != META[k]['rows']: bad.append((k, v['rows'], META[k]['rows']))
    return (not bad), 'streamed counts == footer counts' if not bad else bad
def t_a05():
    k = min(TIMESUM, key=lambda x: META[x]['rows']); pt = TIMESUM[k]['primary_time_column']
    if not pt: return True, 'n/a (no time column)'
    if META[k]['rows'] > 25_000_000: return True, 'n/a (too large to recompute)'
    s = pd.read_parquet(TG / k, columns=[pt])[pt]; opened(TG / k)
    v = pd.to_datetime(s, utc=True).astype('int64').to_numpy() if pd.api.types.is_datetime64_any_dtype(s) else None
    if v is None:
        st = {}; v = to_ns(pa.array(s.to_numpy()), st)
    inv = int((np.diff(v) < 0).sum()); ok = inv == TIMESUM[k]['inversions_(next<prev)'] and pd.Timestamp(int(v.min()), unit='ns', tz='UTC').isoformat() == TIMESUM[k]['tmin']
    return ok, f'{k}: inversions {inv} vs {TIMESUM[k]["inversions_(next<prev)"]}'
def t_a06():
    sym = next((s for s, r in REPLAY.items() if r.get('status') == 'OK'), None)
    if not sym: return True, 'n/a (no successful replay)'
    again = replay_symbol(sym); a, b = REPLAY[sym], again
    return all(a.get(x) == b.get(x) for x in ('events_replayed', 'book_samples', 'crossed_book_share_of_samples', 'spread_bps_median')), f'{sym} replay repeated identically'
def t_a08():
    now = {str(p.relative_to(L1)): {'size': p.stat().st_size, 'mtime_ns': p.stat().st_mtime_ns} for p in sorted(L1.rglob('*')) if p.is_file()} if L1.is_dir() else {}
    return now == PIN_L1, f'Layer 1 {len(now)} files unchanged'
def t_a09(): return (not SECTION_ERRORS), 'no section errors' if not SECTION_ERRORS else f'section errors: {list(SECTION_ERRORS)}'
for nm, fn in (('A01 dataset folder unchanged (sha256)', t_a01), ('A02 no out-of-scope (SOLUSDT) file opened', t_a02), ('A03 report is valid JSON', t_a03), ('A04 streamed row counts == parquet footer counts', t_a04), ('A05 streamed time stats == pandas recompute (smallest file)', t_a05), ('A06 replay deterministic', t_a06), ('A08 Layer 1 unchanged', t_a08), ('A09 all audit sections completed', t_a09)): tcase(nm, fn)

PASS A01 dataset folder unchanged (sha256) | 63 files re-hashed identical
PASS A02 no out-of-scope (SOLUSDT) file opened | 41 open events, none out-of-scope
PASS A03 report is valid JSON | report serialises (72 KiB)
PASS A04 streamed row counts == parquet footer counts | streamed counts == footer counts
PASS A05 streamed time stats == pandas recompute (smallest file) | snapshots/binance/ETHUSDT/2026-06.parquet: inversions 0 vs 0
PASS A06 replay deterministic | BTCUSDT replay repeated identically
PASS A08 Layer 1 unchanged | Layer 1 400 files unchanged
PASS A09 all audit sections completed | no section errors


In [ ]:
def fnum(x, f='{:,.4g}'):
    return 'n/a' if x is None else (f.format(x) if isinstance(x, (int, float, np.integer, np.floating)) else str(x))
L = []; P = L.append
P('=' * 100); P('FINAL AUDIT SHARE BLOCK  -  TGNN dataset audit v2 (READ-ONLY)'); P('=' * 100)
P(f'run: {RUN_STARTED}  |  dataset: MaximumLeverage/crypto-lob-stream  |  folder: {TG}')
P(f'\n[1] FILES / SIZES / SCHEMAS / ROWS  (observed {len(INVDF)} files, {tot/1e9:.3f} GB; stated 18 files / ~1.605 GB)')
for r in INV:
    m = META.get(r['path']); P(f"  {r['path']} | {r['size_mb']} MB | {r['symbol_from_path']} | {r['category_from_path']} | {'IN SCOPE' if r['in_scope'] else 'OUT OF SCOPE (not opened)'}" + (f" | rows {m['rows']:,} | layout {LAYOUT[r['path']]} | cols {list(m['schema'].items())[:14]}" if m else ''))
P('\n[2] TIMESTAMPS')
for k, v in TIMESUM.items():
    P(f"  {k}: col {v.get('primary_time_column')} unit {v.get('unit_detected')} | {v.get('tmin')} -> {v.get('tmax')} ({fnum(v.get('span_hours'))} h) | days {v.get('utc_days_with_data')} | inversions {v.get('inversions_(next<prev)')} ({fnum(v.get('inversion_share'))}) | equal-consecutive {fnum(v.get('equal_consecutive_share'))} | median gap {fnum(v.get('gap_ms_median_approx'))} ms, p99 {fnum(v.get('gap_ms_p99_approx'))} ms | gaps {v.get('gaps')} | hours with data {v.get('hours_with_data')}/{v.get('hours_in_span')}")
    if v.get('largest_gaps_s'): P(f"      largest gaps (s, start): {v['largest_gaps_s'][:3]}")
for k, v in IDSUM.items(): P(f'  update-ID continuity {k}: {v}')
P('\n[2b] CAPTURE SEGMENTS (depth messages chained by update ID; break if U != prev_u+1 or time gap > 30 s)')
for k, v in SEGSUM.items():
    if not k.startswith('_'): P(f'  {k}: ' + json.dumps(v, default=jsonable))
P('  snapshot anchoring: ' + json.dumps(SEGSUM.get('_snapshot_anchoring'), default=jsonable))
P('\n[3] ORDER-BOOK RECONSTRUCTION (README: prune to depth 1000 after every update; snapshot last_update_id is the replay anchor)')
for sym, R in REPLAY.items():
    P(f"  {sym}: {R.get('status')} | anchor mode: {R.get('anchor_mode')} | anchorable snapshots {R.get('anchorable_snapshots')}/{R.get('snapshots_total')} | depth {R.get('depth_file')} | segment {R.get('segment_start')} ({fnum(R.get('segment_duration_s'))} s, {R.get('segment_msgs')} msgs) | events {R.get('events_replayed')} | init levels {R.get('init_snapshot_levels')} / first-msg rows {R.get('init_rows_from_first_message')} | side enc: {R.get('depth_side_encoding')}")
    P(f"     crossed-book share {fnum(R.get('crossed_book_share_of_samples'))} | phantom-removal rate {fnum(R.get('phantom_removal_rate_(removal_of_level_not_in_book)'))} | delta-known share {fnum(R.get('delta_known_share_of_events'))} | levels/side median {R.get('levels_per_side_median')} | spread bps median {fnum(R.get('spread_bps_median'))} | events/s {fnum(R.get('events_per_second'))} | independent in-segment checkpoints {R.get('independent_checkpoint_snapshots_in_segment')}")
for k, w in WIDE.items(): P(f'  wide book {k}: {w}')
P('  verdict per symbol: ' + str(gv(VER, 'criteria', 'reconstruction')))
P('\n[4] ORDER-LEVEL vs PRICE-LEVEL: ' + str(gv(ORDERLVL, '_summary', 'conclusion')))
P('\n[5] SIGNALS FOR SPOOFING-LIKE / LAYERING-LIKE / LIQUIDITY-WITHDRAWAL ANALYSIS (heuristic, descriptive)')
for sym, o in SPOOF.items():
    P(f"  {sym}: window {fnum(o.get('window_hours'))} h | reductions explained by trades {fnum(o.get('reduction_explained_by_trades_share'))} (unexplained/cancel-like {fnum(o.get('reduction_unexplained_share_(cancel-like_or_timing)'))}) | big adds {o.get('big_adds')} (q99.9 >= {fnum(o.get('big_add_threshold_q999'))}) | vanish<=10s {fnum(o.get('vanish_within_10s_share'))} | spoof-like pattern {o.get('spoof_like_count')} ({fnum(o.get('spoof_like_per_hour_(on_candidates_examined)'))}/h) | layering-like clusters {o.get('layering_like_clusters_(>=3_levels_same_side_same_second)')} | withdrawal-like {o.get('liquidity_withdrawal_like_episodes_(depth25 falls>50pct in 60s, |mid move|<10bps)')} ({fnum(o.get('liquidity_withdrawal_like_per_hour'))}/h)")
P('  legitimate signals: price-level qty changes, trade-matched reductions, depth within bps of mid, level co-movement, spread/imbalance; NOT available unless order IDs exist: order lifetime, queue position, participant identity.')
P('\n[6] VERIFIED MANIPULATION LABELS: ' + str(LABELS.get('verdict')))
P('\n[7] PROVENANCE / LICENCE: text files shipped: ' + str(list(PROV['text_files'])) + ' | licence lines: ' + str(PROV['license_lines'][:4]) + ' | urls: ' + str(PROV['urls'][:3]) + ' | parquet writers: ' + str(PROV.get('created_by')) + ' | custom parquet metadata: ' + str(PROV.get('parquet_custom_metadata_keys')))
rd = PROV['text_files'].get('README.md'); P('  README.md excerpt (first 5000 chars):\n' + (rd[:5000] if rd else 'n/a'))
P('\n[8] PROPOSED CAUSAL GRAPH DESIGN'); P('  ' + json.dumps({k: GRAPH[k] for k in ('scope', 'time', 'snapshot_unit')}, indent=1)); P('  nodes: ' + json.dumps(GRAPH['nodes'])); P('  edges: ' + json.dumps(GRAPH['edges'])); P('  baselines: ' + json.dumps(GRAPH['must_have_baselines'])); P('  not representable: ' + GRAPH['not_representable'])
P('\n[9] COMPATIBILITY WITH 2024 LAYER 1'); P('  ' + json.dumps({k: v for k, v in COMPAT.items() if k != 'structural_differences'}, default=jsonable)[:1500])
for x in COMPAT.get('structural_differences', []): P('   - ' + x)
P('\n[10] SUFFICIENCY'); P('  criteria: ' + json.dumps(VER.get('criteria'), default=jsonable)[:1800]); P('  can support: ' + json.dumps(VER.get('can_support')))
P('\nUNCERTAINTIES'); [P('  - ' + u) for u in VER.get('uncertainties', [])]
P('\nRECOMMENDATION: ' + str(VER.get('recommendation')))
P('\nAUDIT TESTS: ' + f"{sum(t[1] for t in TESTS)}/{len(TESTS)} passed"); [P(f"  {'PASS' if ok else 'FAIL'} {n} | {m}") for n, ok, m in TESTS]
if SECTION_ERRORS: P('\nSECTION ERRORS (paste these back):'); [P(f'  {k}:\n{v}') for k, v in SECTION_ERRORS.items()]
P('\nNothing was trained. No dataset, model or protocol was modified. WAITING FOR YOUR APPROVAL before any TGNN protocol or training.'); P('=' * 100)
SHARE = '\n'.join(L)
for sub in ('reports',): (STAGE / sub).mkdir(parents=True, exist_ok=True)
REPORT['tests'] = TESTS; REPORT['section_errors'] = SECTION_ERRORS; REPORT['opened_files'] = sorted(set(OPENED))
dump_json({k: v for k, v in REPORT.items()}, STAGE / 'reports' / 'audit_report.json'); (STAGE / 'reports' / 'share_block.txt').write_text(SHARE, encoding='utf-8'); INVDF.to_csv(STAGE / 'reports' / 'file_inventory.csv', index=False)
dump_json({'dataset_pin_sha256': PIN_TG, 'run_started': RUN_STARTED, 'finished': UTC()}, STAGE / 'reports' / 'dataset_pins.json')
MAN = {str(p.relative_to(STAGE)): sha256_file(p) for p in sorted(STAGE.rglob('*')) if p.is_file()}; dump_json(MAN, STAGE / 'manifest.json')
okm = all(sha256_file(STAGE / k) == v for k, v in MAN.items()); all_ok = all(t[1] for t in TESTS) and okm
if all_ok: os.rename(STAGE, OUT_FINAL); print('\nSAVED ->', OUT_FINAL)
else: print('\n!! Some tests FAILED -> staging kept at', STAGE, '(not renamed). Share this block anyway.')
print(SHARE); print(f'A07 manifest verified: {okm} ({len(MAN)} files)')


SAVED -> /content/drive/MyDrive/research datasets/audit_tgnn_dataset_v2
FINAL AUDIT SHARE BLOCK  -  TGNN dataset audit v2 (READ-ONLY)
run: 2026-10-12T05:04:40.307087+00:00  |  dataset: MaximumLeverage/crypto-lob-stream  |  folder: /content/drive/MyDrive/research datasets/TGNN_Public_OrderBook_Dataset

[1] FILES / SIZES / SCHEMAS / ROWS  (observed 63 files, 1.724 GB; stated 18 files / ~1.605 GB)
  .cache/huggingface/.gitignore | 0.0 MB | None | hf_cache_metadata | OUT OF SCOPE (not opened)
  .cache/huggingface/CACHEDIR.TAG | 0.0 MB | None | hf_cache_metadata | OUT OF SCOPE (not opened)
  .cache/huggingface/download/.gitattributes.lock | 0.0 MB | None | hf_cache_metadata | OUT OF SCOPE (not opened)
  .cache/huggingface/download/.gitattributes.metadata | 0.0 MB | None | hf_cache_metadata | OUT OF SCOPE (not opened)
  .cache/huggingface/download/README.md.lock | 0.0 MB | None | hf_cache_metadata | OUT OF SCOPE (not opened)
  .cache/huggingface/download/README.md.metadata | 0.0 MB | None |